In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

from core.noise_scheduler import DDPMScheduler
from core.policy import DiffusionPolicy
from core.network import DummyNoisePredictor # 你可以自己写一个简单的多层MLP占位

def rel_error(x, y):
    """返回相对误差"""
    return torch.max(torch.abs(x - y) / (torch.maximum(torch.tensor(1e-8), torch.abs(x) + torch.abs(y))))

In [ ]:
scheduler = DDPMScheduler(num_train_timesteps=100)
torch.manual_seed(42)

# 模拟一个 2 个样本，序列长度为 8，自由度为 20（如灵巧手）的动作张量
dummy_actions = torch.ones((2, 8, 20)) 
dummy_noise = torch.randn_like(dummy_actions)
timesteps = torch.tensor([10, 90]) # 测试早期和晚期的 t

noisy_samples = scheduler.add_noise(dummy_actions, dummy_noise, timesteps)

# ======= 期望的输出 (预先计算好的正确值片段) =======
expected_mean_t10 = 0.9472
expected_std_t10 = 0.3205

# 运行你的实现与期望结果比对
assert noisy_samples.shape == (2, 8, 20), "Shape mismatch!"
error = rel_error(noisy_samples[0].mean(), torch.tensor(dummy_actions[0].mean() * expected_mean_t10 + dummy_noise[0].mean() * expected_std_t10))
print(f"Add Noise Error (t=10): {error.item():.2e}")
if error < 1e-4:
    print("✅ 前向加噪公式实现正确！")
else:
    print("❌ 加噪公式有误，请检查公式实现。")

In [ ]:
class DummyPredictor(torch.nn.Module):
    def forward(self, x, t, cond): return x * 0.5 # 仅做维度测试

policy = DiffusionPolicy(DummyPredictor(), scheduler)
dummy_obs = torch.randn((2, 2, 64)) # Obs horizon=2, feat_dim=64

loss = policy.compute_loss(dummy_actions, dummy_obs)
print(f"Training Loss returned a scalar: {loss.item():.4f}")
print("✅ Loss 计算结构正确！")

In [ ]:
torch.manual_seed(2026)
generated_actions = policy.conditional_sample(dummy_obs, pred_horizon=8, action_dim=20)

assert generated_actions.shape == (2, 8, 20), "Generated shape incorrect"
print("✅ 推理循环跑通！")

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(generated_actions[0, :, 0].cpu().numpy(), label="Joint 0")
plt.plot(generated_actions[0, :, 1].cpu().numpy(), label="Joint 1")
plt.title("Sampled Action Trajectories (Denormalized)")
plt.xlabel("Timestep (Pred Horizon)")
plt.ylabel("Joint Angle / Position")
plt.legend()
plt.show()